# Precision & Recall: LLM Content Moderation

This notebook walks through how to evaluate an LLM-based classifier using **precision** and **recall** — two of the most important metrics for content moderation and classification tasks.

## The Dataset
We're working with app store reviews that have been **manually labeled** by humans for three types of harmful content:
- `racism`
- `bullying`  
- `sexual`

Reviews that have been labeled (value is `0` or `1`) form the **golden set** — our ground truth.

## The Goal
We'll build an LLM-based classifier that predicts whether a review contains unwanted sexual content, then measure how well it performs against the human labels.

---

## Key Concepts

**Precision** answers: *"Of everything the model flagged, what fraction was actually harmful?"*
- High precision = few false alarms
- `precision = true_positives / (true_positives + false_positives)`

**Recall** answers: *"Of all the actually harmful content, what fraction did the model catch?"*
- High recall = few misses
- `recall = true_positives / (true_positives + false_negatives)`

There's usually a trade-off: tuning your prompt to catch more bad content (higher recall) tends to also flag more innocent content (lower precision), and vice versa. Understanding this trade-off is the whole point of this exercise.


---
## Step 1: Read the Data

In [ ]:
import pandas as pd

df = pd.read_csv("reviews-actual.csv", low_memory=False)

print(f"Total rows: {len(df)}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head(10)


---
## Step 2: Load the Golden Set

The **golden set** is a curated sample of reviews that have been manually labeled by humans for four types of harmful content:
- `sexually_explicit`
- `unwanted_sexual`
- `racism`
- `bullying`

A value of `1` means the review contains that type of content. A value of `0` means a human reviewed it and decided it does not.

We load it directly here — no filtering needed.


In [ ]:
golden = pd.read_csv("review-sample.csv")

# Combine title and body into a single review_text column; handle missing values
golden["review_text"] = golden["title"].fillna("") + " " + golden["body"].fillna("")
golden["review_text"] = golden["review_text"].str.strip()

print(f"Total rows in golden set: {len(golden)}")

# Pivot table showing how many 0s and 1s for each label category
label_cols = ["sexually_explicit", "unwanted_sexual", "racism", "bullying"]
summary = pd.DataFrame({
    col: golden[col].value_counts().reindex([0, 1], fill_value=0)
    for col in label_cols
})
summary.index = ["0 (no)", "1 (yes)"]
summary


---
## Step 3: Write the LLM Prompt

This is the part you'll experiment with! We're going to ask an LLM to read each review and decide whether it contains unwanted sexual content.

The prompt below is a starting point — it's intentionally simple. After you see the precision/recall results, you can come back and tweak the prompt to try to improve the scores.

**Things to experiment with:**
- How specific is your definition of "sexual content"?
- Do you ask for just a yes/no, or ask the model to reason first?
- Do you give examples?
- Do you set a strict or lenient threshold?

We're using [OpenRouter](https://openrouter.ai/) so you can swap in different models without changing the code structure.


In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
# MODEL = "openai/gpt-5.4-mini"  # ✏️ Change this to try different models
MODEL = "google/gemini-2.5-flash"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"Using model: {MODEL}")
print(f"API key loaded: {'yes' if OPENROUTER_API_KEY else 'NO - check your .env file'}")


### Structured Outputs

Instead of asking the model to reply with "YES" or "NO" and trying to parse that text, we use **structured outputs** — we give the model a schema and it fills it in like a form.

We define two fields:
- `is_sexual` — a boolean (`True`/`False`)
- `reason` — a short string explaining the decision

The schema is passed directly to the API, so the model is *constrained* to return exactly this shape — not just asked nicely.


In [ ]:
class ModerationResult(BaseModel):
    is_sexual: bool  # True if the review contains unwanted sexual comments
    reason: str      # Short explanation of why the model made this decision


### The Prompt

This is the part you'll experiment with! Write a system prompt and a user prompt that together tell the model what to look for.

**Things to try:**
- How specific is your definition of "unwanted sexual comments"?
- Do you ask the model to reason before deciding?
- Do you give examples of what counts (or doesn't count)?


In [ ]:
# ✏️ Edit these prompts to experiment!

SYSTEM_PROMPT = """You are a content moderation assistant.
Your job is to detect whether app store reviews contain unwanted sexual comments.
Respond in JSON with two fields: "is_sexual" (boolean) and "reason" (string)."""

USER_PROMPT = """Does the following app store review contain unwanted sexual comments?

Review: {review_text}"""


In [ ]:
import json

def classify_sexual_content(review_text: str) -> ModerationResult:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(review_text=review_text)},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "ModerationResult",
                "strict": True,
                "schema": ModerationResult.model_json_schema(),
            },
        },
        max_tokens=500,
        temperature=0,
    )
    data = json.loads(response.choices[0].message.content)
    return ModerationResult(**data)


### Sanity Check

Before running on all 331 reviews, test the function on one review to make sure it works.


In [ ]:
test_review = "Great app for meeting people!"
result = classify_sexual_content(test_review)

print(f"Review:    {test_review}")
print(f"is_sexual: {result.is_sexual}")
print(f"reason:    {result.reason}")


---
## Step 4: Run the Classifier on the Golden Set

Now we run `classify_sexual_content()` on every review in `golden_sexual` and store the results in a new column called `sexual_llm`. You'll see a progress bar as it runs.

> **Note:** We're only running this on rows that were actually labeled for sexual content — that's all we need for evaluation. Running it on all 56k rows would cost money and take a long time, and we don't have ground truth labels for those rows anyway.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def _run(idx_and_review):
    idx, review_text = idx_and_review
    for attempt in range(3):
        try:
            result = classify_sexual_content(str(review_text))
            return idx, result.is_sexual, result.reason
        except Exception:
            if attempt == 2:
                raise


Now run it on every review. This sends all the API calls in parallel and shows a progress bar as results come in.


In [ ]:
reviews = list(golden["review_text"].items())
unwanted_sexual_llm, reasons = {}, {}

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(_run, item): item for item in reviews}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying reviews"):
        idx, is_sexual, reason = future.result()
        unwanted_sexual_llm[idx] = int(is_sexual)
        reasons[idx] = reason

golden["unwanted_sexual_llm"] = pd.Series(unwanted_sexual_llm)
golden["reason"] = pd.Series(reasons)

print(f"LLM flagged as sexual: {golden['unwanted_sexual_llm'].sum()}")
print(f"LLM said not sexual:   {(golden['unwanted_sexual_llm'] == 0).sum()}")
golden[["body", "unwanted_sexual", "unwanted_sexual_llm", "reason"]].head(10)


---
## Step 5: Calculate Precision and Recall

Now we compare the LLM's predictions (`sexual_llm`) against the human labels (`sexual`) to compute precision and recall.

First, let's understand the four possible outcomes for each review:

| | Human says: Sexual | Human says: Not Sexual |
|---|---|---|
| **LLM says: Sexual** | ✅ True Positive (TP) | ❌ False Positive (FP) |
| **LLM says: Not Sexual** | ❌ False Negative (FN) | ✅ True Negative (TN) |

- **True Positives (TP):** Correctly caught harmful content
- **False Positives (FP):** Flagged innocent reviews (false alarms)  
- **False Negatives (FN):** Missed actual harmful content
- **True Negatives (TN):** Correctly passed innocent reviews

Then:
- **Precision** = TP / (TP + FP) — "How accurate are the flags?"
- **Recall** = TP / (TP + FN) — "How much harmful content did we catch?"


In [ ]:
human   = golden["unwanted_sexual"]  # ground truth (0 or 1)
llm     = golden["unwanted_sexual_llm"]       # LLM predictions (0 or 1)

# Count the four outcome types
tp = int(((human == 1) & (llm == 1)).sum())  # both said yes
fp = int(((human == 0) & (llm == 1)).sum())  # LLM said yes, human said no
fn = int(((human == 1) & (llm == 0)).sum())  # LLM said no, human said yes
tn = int(((human == 0) & (llm == 0)).sum())  # both said no

print("Confusion Matrix:")
print(f"  True Positives  (TP): {tp:>4}  — correctly flagged as unwanted sexual comments")
print(f"  False Positives (FP): {fp:>4}  — incorrectly flagged (false alarms)")
print(f"  False Negatives (FN): {fn:>4}  — missed actual unwanted sexual comments")
print(f"  True Negatives  (TN): {tn:>4}  — correctly passed as clean")

try:
    precision = tp / (tp + fp)
except ZeroDivisionError:
    precision = 0.0

try:
    recall = tp / (tp + fn)
except ZeroDivisionError:
    recall = 0.0

try:
    f1 = 2 * (precision * recall) / (precision + recall)
except ZeroDivisionError:
    f1 = 0.0

print(f"\n{'='*40}")
print(f"  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")
print(f"  F1 Score:  {f1:.2%}")
print(f"{'='*40}")
print(f"\nInterpretation:")
print(f"  - The LLM flagged {tp + fp} reviews as containing unwanted sexual comments")
print(f"  - Of those, {tp} matched the human label ({precision:.0%} precision)")
print(f"  - Out of {tp + fn} reviews humans labeled unwanted_sexual=1, the LLM caught {tp} ({recall:.0%} recall)")


---
## Dig Deeper: Inspect Errors

It's useful to actually *read* the reviews that the model got wrong. Looking at false positives and false negatives often gives you intuition for how to improve the prompt.


In [ ]:
pd.set_option("display.max_colwidth", None)

# False Positives: LLM flagged it, but humans said it was fine
false_positives = golden[(human == 0) & (llm == 1)][["review_text", "unwanted_sexual", "unwanted_sexual_llm", "reason"]]
print(f"False Positives ({len(false_positives)} reviews the LLM incorrectly flagged):")
false_positives


In [ ]:
# False Negatives: Humans flagged it, but LLM missed it
false_negatives = golden[(human == 1) & (llm == 0)][["review_text", "unwanted_sexual", "unwanted_sexual_llm", "reason"]]
print(f"False Negatives ({len(false_negatives)} reviews the LLM missed):")
false_negatives
